# E3h Rot4 TTA Colab Runner

Thin Colab launcher for the E3h validation-only rot4 test-time augmentation diagnostic. Core logic stays in `src/` and `scripts/`.

This notebook does not load or evaluate the test split. Outputs are isolated under `/content/drive/MyDrive/dl-final-artifact/e3h_tta_rot4/`.

## 1. Mount Drive, clone/pull repo, restore HAM10000 bundle

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=True)

DRIVE_ROOT = "/content/drive/MyDrive/dl-final-artifact"
E3H_DRIVE_ROOT = f"{DRIVE_ROOT}/e3h_tta_rot4"
E3H_INPUT_BUNDLE = f"{E3H_DRIVE_ROOT}/e3h_tta_inputs.tar"
REPO_URL = "https://github.com/KasimDeliaci/dl-final-project.git"
REPO_DIR = "/content/dl-final-project"

# T4-safe default. If Colab gives L4/A100 and memory is stable, raise to 192 or 256.
E3H_BATCH_SIZE = 128
E3H_NUM_WORKERS = 2
E3H_IDENTITY_TOLERANCE = "1e-3"

!mkdir -p "$DRIVE_ROOT" "$E3H_DRIVE_ROOT"
!if [ ! -d "$REPO_DIR/.git" ]; then git clone "$REPO_URL" "$REPO_DIR"; fi
%cd /content/dl-final-project
!git pull --ff-only
!git rev-parse --short HEAD

BUNDLE_CANDIDATES = [
    f"{DRIVE_ROOT}/ham10000_colab_bundle.tar",
    "/content/drive/MyDrive/ham10000_colab_bundle.tar",
    "/content/drive/MyDrive/Colab Notebooks/ham10000_colab_bundle.tar",
]
bundle = next((path for path in BUNDLE_CANDIDATES if Path(path).exists()), None)
print("HAM10000 bundle:", bundle or "not found")
if bundle:
    !tar -xf "$bundle" -C /content/dl-final-project
else:
    print("Upload/copy ham10000_colab_bundle.tar before full E3h run.")

print("E3h Drive root:", E3H_DRIVE_ROOT)
print("E3h input bundle fallback:", E3H_INPUT_BUNDLE)
print("E3h batch size:", E3H_BATCH_SIZE)


## 2. Install dependencies

In [ ]:
!pip -q install uv
!uv sync --dev

## 3. Verify GPU and canonical split

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("E3h full TTA should run on a Colab GPU runtime. Enable GPU first.")

In [ ]:
!PYTHONPATH=src uv run python scripts/audit_ham10000.py \
  --config configs/dataset/selected_dataset.yaml
!PYTHONPATH=src uv run python scripts/make_lesion_split.py \
  --config configs/dataset/selected_dataset.yaml

import pandas as pd

EXPECTED_SPLIT_ROWS = {"train": 7008, "val": 1504, "test": 1503}
for split, expected in EXPECTED_SPLIT_ROWS.items():
    path = Path("data/splits") / f"{split}.csv"
    actual = len(pd.read_csv(path))
    print(split, actual)
    if actual != expected:
        raise RuntimeError(f"{split}.csv has {actual} rows; expected {expected}.")

## 4. Restore required input artifacts from Drive

Required inputs come from previous validation-only experiments. This cell restores only the needed operational artifacts into local Colab storage.

In [ ]:
import subprocess

DRIVE_ARTIFACT_ROOT = Path(DRIVE_ROOT) / "artifacts"
LOCAL_ARTIFACT_ROOT = Path("artifacts")

input_bundle = Path(E3H_INPUT_BUNDLE)
if input_bundle.exists():
    print(f"Restoring optional E3h input bundle: {input_bundle}")
    subprocess.run(["tar", "-xf", str(input_bundle), "-C", str(Path.cwd())], check=True)
else:
    print(f"No E3h input bundle found at {input_bundle}; using Drive artifact tree.")


def sync_dir(source: Path, dest: Path, *, required: bool = True) -> None:
    if not source.exists():
        message = f"missing source: {source}"
        if dest.exists():
            print(f"WARNING: {message}; keeping existing local {dest}")
            return
        if required:
            raise FileNotFoundError(message)
        print("WARNING:", message)
        return
    dest.mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync", "-a", f"{source}/", f"{dest}/"], check=True)


def matching_run_configs(root: Path, pattern: str) -> list[Path]:
    return [path for path in sorted(root.glob(pattern)) if path.is_file()]


def sync_matching_dirs(
    source_root: Path,
    pattern: str,
    dest_root: Path,
    *,
    required_count: int | None = None,
) -> None:
    matches = matching_run_configs(source_root, pattern)
    local_matches = matching_run_configs(dest_root, pattern)
    if required_count is not None and len(matches) != required_count:
        if len(local_matches) == required_count:
            print(
                f"WARNING: {pattern}: Drive has {len(matches)} configs under "
                f"{source_root}; keeping {len(local_matches)} local configs."
            )
            return
        raise RuntimeError(
            f"{pattern}: expected {required_count} configs, found {len(matches)} "
            f"under {source_root} and {len(local_matches)} local configs under {dest_root}"
        )
    dest_root.mkdir(parents=True, exist_ok=True)
    for config_path in matches:
        run_dir = config_path.parent
        target = dest_root / run_dir.name
        target.mkdir(parents=True, exist_ok=True)
        subprocess.run(["rsync", "-a", f"{run_dir}/", f"{target}/"], check=True)


sync_dir(
    DRIVE_ARTIFACT_ROOT / "checkpoints/ham10000/finetuned",
    LOCAL_ARTIFACT_ROOT / "checkpoints/ham10000/finetuned",
)
sync_dir(
    DRIVE_ARTIFACT_ROOT / "features/ham10000/finetuned",
    LOCAL_ARTIFACT_ROOT / "features/ham10000/finetuned",
)
# E3f/E3h mixed source is derived: frozen ViT + fine-tuned Swin/BEiT.
# Prefer a previously materialized source, but rebuild it in Colab when the Drive tree lacks it.
sync_dir(
    DRIVE_ARTIFACT_ROOT / "features/ham10000/frozen_vit_finetuned_swin_beit",
    LOCAL_ARTIFACT_ROOT / "features/ham10000/frozen_vit_finetuned_swin_beit",
    required=False,
)
sync_dir(
    DRIVE_ARTIFACT_ROOT / "features/ham10000/frozen/vit_b16",
    LOCAL_ARTIFACT_ROOT / "features/ham10000/frozen/vit_b16",
    required=False,
)


def ensure_mixed_feature_source() -> None:
    mixed_root = LOCAL_ARTIFACT_ROOT / "features/ham10000/frozen_vit_finetuned_swin_beit"
    mixed_backbones = ["vit_b16", "swin_tiny", "beit_base"]
    required_val_files = [mixed_root / backbone / "val.pt" for backbone in mixed_backbones]
    if all(path.exists() for path in required_val_files):
        print("Mixed feature source already available.")
        return

    source_dirs = {
        "vit_b16": LOCAL_ARTIFACT_ROOT / "features/ham10000/frozen/vit_b16",
        "swin_tiny": LOCAL_ARTIFACT_ROOT / "features/ham10000/finetuned/swin_tiny",
        "beit_base": LOCAL_ARTIFACT_ROOT / "features/ham10000/finetuned/beit_base",
    }
    for backbone, source_dir in source_dirs.items():
        if not (source_dir / "val.pt").exists():
            raise FileNotFoundError(
                f"Cannot build mixed source; missing {source_dir / 'val.pt'}. "
                "Upload e3h_tta_inputs.tar or restore the base frozen/finetuned caches to Drive."
            )
        dest_dir = mixed_root / backbone
        dest_dir.mkdir(parents=True, exist_ok=True)
        subprocess.run(["rsync", "-a", f"{source_dir}/", f"{dest_dir}/"], check=True)
    print("Built mixed feature source from frozen ViT and fine-tuned Swin/BEiT caches.")


ensure_mixed_feature_source()

sync_matching_dirs(
    DRIVE_ARTIFACT_ROOT / "runs",
    "*e3d_triple_metadata_film_e3d_metadata_fusion_seed*/run_config.json",
    LOCAL_ARTIFACT_ROOT / "runs",
    required_count=5,
)
sync_matching_dirs(
    DRIVE_ARTIFACT_ROOT / "runs",
    "*e3d_triple_metadata_gated_backbone_e3d_metadata_fusion_seed*/run_config.json",
    LOCAL_ARTIFACT_ROOT / "runs",
    required_count=5,
)
sync_matching_dirs(
    DRIVE_ARTIFACT_ROOT / "runs",
    "*e3d_triple_metadata_gated_backbone_e3f_mixed_adaptation_seed*/run_config.json",
    LOCAL_ARTIFACT_ROOT / "runs",
    required_count=5,
)
sync_dir(
    DRIVE_ARTIFACT_ROOT / "runs/e3g_prediction_ensemble",
    LOCAL_ARTIFACT_ROOT / "runs/e3g_prediction_ensemble",
)
print("Required E3h input artifacts restored.")


## 5. Input artifact integrity check

In [ ]:
import json

import torch

feature_checks = []
for source in ["finetuned", "frozen_vit_finetuned_swin_beit"]:
    for backbone in ["vit_b16", "swin_tiny", "beit_base"]:
        path = Path("artifacts/features/ham10000") / source / backbone / "val.pt"
        if not path.exists():
            feature_checks.append(
                {
                    "source": source,
                    "backbone": backbone,
                    "status": "missing",
                    "path": str(path),
                }
            )
            continue
        payload = torch.load(path, map_location="cpu", weights_only=False)
        features = payload["features"]
        feature_checks.append(
            {
                "source": source,
                "backbone": backbone,
                "rows": int(features.shape[0]),
                "feature_dim": int(features.shape[1]),
                "finite": bool(torch.isfinite(features).all()),
                "path": str(path),
            }
        )
feature_checks = pd.DataFrame(feature_checks)
display(feature_checks)
if (feature_checks.get("status", "") == "missing").any():
    raise RuntimeError("Missing required feature caches.")
if not (feature_checks["rows"] == 1504).all():
    raise RuntimeError("Feature cache row-count mismatch.")
if not feature_checks["finite"].all():
    raise RuntimeError("Feature cache contains non-finite values.")

run_groups = {
    "e3d_film": "*e3d_triple_metadata_film_e3d_metadata_fusion_seed*/run_config.json",
    "e3d_gated": (
        "*e3d_triple_metadata_gated_backbone_e3d_metadata_fusion_seed*/run_config.json"
    ),
    "e3f_mixed_gated": (
        "*e3d_triple_metadata_gated_backbone_e3f_mixed_adaptation_seed*/run_config.json"
    ),
}
for name, pattern in run_groups.items():
    runs = sorted(Path("artifacts/runs").glob(pattern))
    seeds = sorted(json.loads(path.read_text())["seed"] for path in runs)
    print(name, len(runs), seeds)
    if seeds != [7, 13, 42, 101, 202]:
        raise RuntimeError(f"Unexpected runs for {name}: {seeds}")
    for config_path in runs:
        run_dir = config_path.parent
        required_files = [
            "model.pt",
            "scaler_stats.npz",
            "metadata_preprocessing.json",
            "predictions.csv",
        ]
        for required in required_files:
            if not (run_dir / required).exists():
                raise FileNotFoundError(run_dir / required)
print("E3h input artifact checks passed.")


## 6. Code verification

In [ ]:
!PYTHONPATH=src uv run ruff check .
!PYTHONPATH=src uv run python -m pytest tests/test_e3h_tta.py

## 7. Smoke run

Run this first. It writes smoke artifacts under `artifacts/runs/e3h_tta_rot4_smoke_n8/` and should complete quickly on GPU.

In [ ]:
!PYTHONUNBUFFERED=1 PYTHONPATH=src uv run python scripts/evaluate_tta_rot4.py \
  --max-samples 8 \
  --batch-size {E3H_BATCH_SIZE} \
  --num-workers {E3H_NUM_WORKERS} \
  --device cuda \
  --identity-tolerance {E3H_IDENTITY_TOLERANCE}


## 8. Full validation-only E3h run

This is the canonical E3h run. It does not load the test split.

In [ ]:
!PYTHONUNBUFFERED=1 PYTHONPATH=src uv run python scripts/evaluate_tta_rot4.py \
  --batch-size {E3H_BATCH_SIZE} \
  --num-workers {E3H_NUM_WORKERS} \
  --device cuda \
  --identity-tolerance {E3H_IDENTITY_TOLERANCE}


## 9. Output integrity check

In [ ]:
from dl_final.evaluation.tta import probabilities_from_frame

RUN_DIR = Path("artifacts/runs/e3h_tta_rot4")
pred_path = RUN_DIR / "predictions_top3_family_equal_tta_rot4.csv"
if not pred_path.exists():
    raise FileNotFoundError(pred_path)
pred = pd.read_csv(pred_path)
print("prediction rows", len(pred))
if len(pred) != 1504:
    raise RuntimeError(f"E3h prediction dump has {len(pred)} rows; expected 1504.")
if set(pred["split"].astype(str)) != {"val"}:
    raise RuntimeError("E3h predictions contain non-validation rows.")
_ = probabilities_from_frame(pred, ["akiec", "bcc", "bkl", "df", "nv", "mel", "vasc"])

identity = pd.read_csv(RUN_DIR / "tta_identity_sanity.csv")
display(identity[["family", "seed", "max_abs_probability_delta", "passed"]])
if not identity["passed"].all():
    raise RuntimeError("Identity sanity check failed.")

results = pd.read_csv(RUN_DIR / "tta_ensemble_results.csv")
display(results)
print("E3h output integrity check passed.")

## 10. Display report assets safely

In [ ]:
from IPython.display import Image, Markdown, display

TABLE_ROOT = Path("artifacts/report_assets/tables")
FIGURE_ROOT = Path("artifacts/report_assets/figures")


def safe_display_csv(path: Path) -> None:
    display(Markdown(f"### {path}"))
    if not path.exists():
        display(Markdown("missing"))
        return
    if path.stat().st_size == 0:
        display(Markdown("empty file"))
        return
    try:
        display(pd.read_csv(path))
    except pd.errors.EmptyDataError:
        display(Markdown("empty file"))


for path in sorted(TABLE_ROOT.glob("e3h_tta_rot4*.csv")):
    safe_display_csv(path)
for path in sorted(FIGURE_ROOT.glob("e3h_tta_rot4*.png")):
    display(Markdown(f"### {path}"))
    display(Image(filename=str(path)))

## 11. Sync E3h outputs to Drive

In [ ]:
E3H_ARTIFACT_ROOT = Path(E3H_DRIVE_ROOT) / "artifacts"
E3H_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

sync_dir(Path("artifacts/runs/e3h_tta_rot4"), E3H_ARTIFACT_ROOT / "runs/e3h_tta_rot4")
if Path("artifacts/runs/e3h_tta_rot4_smoke_n8").exists():
    sync_dir(
        Path("artifacts/runs/e3h_tta_rot4_smoke_n8"),
        E3H_ARTIFACT_ROOT / "runs/e3h_tta_rot4_smoke_n8",
        required=False,
    )

sync_specs = [
    (
        Path("artifacts/report_assets/tables"),
        "e3h_tta_rot4*.csv",
        E3H_ARTIFACT_ROOT / "report_assets/tables",
    ),
    (
        Path("artifacts/report_assets/figures"),
        "e3h_tta_rot4*.png",
        E3H_ARTIFACT_ROOT / "report_assets/figures",
    ),
]
for source_root, pattern, dest_root in sync_specs:
    dest_root.mkdir(parents=True, exist_ok=True)
    matches = [path for path in sorted(source_root.glob(pattern)) if path.is_file()]
    if not matches:
        raise RuntimeError(f"No files matched {pattern} under {source_root}")
    for path in matches:
        subprocess.run(["rsync", "-a", str(path), f"{dest_root}/"], check=True)

print(f"E3h artifact sync complete: {E3H_ARTIFACT_ROOT}")
!find "$E3H_DRIVE_ROOT" -maxdepth 4 -type f | sort | tail -80
